# The Gemini Decisions API

A decision model doesn't write. You give it some text and a few questions. Each answer comes back as a
value and a probability, and your code branches on them.

There are three kinds of question:

| Kind | The API calls it | The answer |
|---|---|---|
| Yes or no | `noul` | The chance it is yes, from 0 to 1 |
| Pick one | `choice` | The option it picked |
| Place on a scale | `score` | The level, counted from 0 |

To run this you need `GEMINI_API_KEY` in `.env`, from Google AI Studio.

In [1]:
import os
import time

from dotenv import load_dotenv
from google import genai
from IPython.display import Markdown

load_dotenv()  # reads GEMINI_API_KEY from .env

# v1beta: the Decisions API is in preview, and the library's default version answers 404 for it.
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"], http_options={"api_version": "v1beta"})
MODEL = "gemini-decision-1"

## 1. One question

The smallest call there is: some text and one yes or no question.

In [2]:
response = client.decisions.create(
    model=MODEL,
    input="Hi, can you automate our invoices? We need it this month.",
    questions={
        "real": {"type": "noul", "instructions": "Is this a genuine enquiry from someone who might buy a service?"},
    },
)

answer = response.answers["real"]
print("chance it is yes:", round(answer.noul, 2))
print("tokens in:       ", response.usage.total_input_tokens)
print("tokens out:      ", response.usage.total_output_tokens)

chance it is yes: 0.9
tokens in:        109
tokens out:       0


Look at the tokens out. The model wrote nothing. That is why it is fast and cheap.

## 2. Three kinds of question, one request

Every question about the same text goes in one request. `criteria` tells the model what each answer means.

In [3]:
QUESTIONS = {
    "real": {
        "type": "noul",
        "instructions": "Is this a genuine enquiry from someone who might buy a service from us?",
        "criteria": {
            "true_description": "They describe a problem at their work, or ask about services, prices or a call",
            "false_description": "Spam, someone selling to us, a job application, or text with no meaning",
        },
    },
    "service": {
        "type": "choice",
        "instructions": "Which one of our services fits what they are asking for?",
        "criteria": {
            "Automation audit": "Two weeks. We map the work and find what to automate first. £3,000",
            "Build one automation": "We build, test and deploy one automation. From £12,000",
            "Support": "We watch and fix automations that already exist. £1,500 a month",
            "Not something we do": "Anything else. For example websites, mobile apps, hardware, training",
        },
    },
    "urgency": {
        "type": "score",
        "instructions": "How soon do they need this?",
        "criteria": [
            "No time pressure is mentioned",
            "They want to start in the next few months",
            "It is costing them money or customers now, or they give a deadline within weeks",
        ],
    },
}


def decide(text: str) -> dict:
    """One request. Each answer comes back as (value, how sure the model is)."""
    start = time.perf_counter()
    response = client.decisions.create(model=MODEL, input=text, questions=QUESTIONS)
    result = {name: (getattr(answer, answer.type), answer.confidence) for name, answer in response.answers.items()}
    result["ms"] = round((time.perf_counter() - start) * 1000)
    result["tokens"] = response.usage.total_tokens
    return result


def show(r: dict) -> None:
    for name in QUESTIONS:
        value, sure = r[name]
        value = value if isinstance(value, str) else f"{value:.2f}"
        print(f"{name:8} {value:22} {sure:4.0%} sure")
    print(f"\n{r['ms']} ms, {r['tokens']} tokens")


show(decide("We re-type every delivery note into our system. Can you help us automate that?"))

real     0.98                    98% sure
service  Build one automation    96% sure
urgency  0.18                    85% sure

342 ms, 383 tokens


Each answer has a value, and how sure the model is of it.

- `real` is the chance this is a genuine enquiry.
- `service` is the option it picked.
- `urgency` is a place on the scale: 0 is the first level, 2 is the last.

## 3. Ten enquiries

The same three questions, asked of ten very different messages.

In [4]:
ENQUIRIES = [
    "We re-type every delivery note into our system. Can you help us automate that?",
    "Our invoices are three weeks behind and suppliers are chasing us. Can you automate our invoice processing?",
    "We have four automations from last year and nobody to look after them.",
    "Not sure where to start with automation. What would you suggest?",
    "Can you build us a mobile app?",
    "CHEAP SEO!!! We will get you to number 1 on Google",
    "asdf asdf asdf",
    "Can you help us with our processes?",
    "Hello, quick question about what you do.",
    "We'd like to automate onboarding at some point next year, and maybe get some training too.",
]

results = [(text, decide(text)) for text in ENQUIRIES]


def table(rows: list, *more_headings: str) -> Markdown:
    """One row for each enquiry. A row is (text, answers), plus one value for each extra heading."""
    headings = ["Enquiry", "Real", "Service", "Sure", "Urgency", *more_headings]
    lines = [" | ".join(headings), " | ".join("---" for _ in headings)]
    for text, r, *more in rows:
        (real, _), (service, sure), (urgency, _) = r["real"], r["service"], r["urgency"]
        lines.append(" | ".join([text, f"{real:.2f}", service, f"{sure:.0%}", f"{urgency:.1f}", *more]))
    return Markdown("\n".join(lines))


took = [r["ms"] for _, r in results]
print(f"{len(results)} requests. Fastest {min(took)} ms, slowest {max(took)} ms, about {results[0][1]['tokens']} tokens each.")
table(results)

10 requests. Fastest 209 ms, slowest 353 ms, about 383 tokens each.


Enquiry | Real | Service | Sure | Urgency
--- | --- | --- | --- | ---
We re-type every delivery note into our system. Can you help us automate that? | 0.98 | Build one automation | 96% | 0.2
Our invoices are three weeks behind and suppliers are chasing us. Can you automate our invoice processing? | 0.99 | Build one automation | 78% | 1.9
We have four automations from last year and nobody to look after them. | 0.90 | Support | 99% | 0.3
Not sure where to start with automation. What would you suggest? | 0.89 | Automation audit | 93% | 0.1
Can you build us a mobile app? | 0.93 | Not something we do | 100% | 0.1
CHEAP SEO!!! We will get you to number 1 on Google | 0.01 | Not something we do | 100% | 0.1
asdf asdf asdf | 0.03 | Not something we do | 69% | 0.1
Can you help us with our processes? | 0.82 | Automation audit | 67% | 0.1
Hello, quick question about what you do. | 0.64 | Automation audit | 49% | 0.1
We'd like to automate onboarding at some point next year, and maybe get some training too. | 0.89 | Not something we do | 75% | 0.4

Three things to notice:

- The spam and the nonsense score close to 0 on `real`.
- The mobile app is a real enquiry, for something we don't sell.
- The last few are vague, and the model says so: look at the `Sure` column.

## 4. Your code sets the thresholds

The model gives probabilities. It doesn't decide what happens next. These three lines do.

This cell makes no requests. It reuses the answers from above. Change a number, run it again, and the
routing changes at once.

In [5]:
SPAM_BELOW = 0.1  # drop an enquiry only if it is under 10% likely to be real
SURE = 0.6        # under 60% sure which service fits? Then let the writing model pick
URGENT = 1.5      # on the 0 to 2 urgency scale


def route(r: dict) -> str:
    (real, _), (service, sure), (urgency, _) = r["real"], r["service"], r["urgency"]
    if real < SPAM_BELOW:
        return "Save as spam. Nobody is pinged"
    flag = "URGENT. " if urgency >= URGENT else ""
    if sure < SURE:
        return flag + "Write a proposal. The writing model picks the service"
    if service == "Not something we do":
        return flag + "A person replies. No model writes anything"
    return flag + f"Write a proposal for: {service}"


table([(text, r, route(r)) for text, r in results], "What the code does")

Enquiry | Real | Service | Sure | Urgency | What the code does
--- | --- | --- | --- | --- | ---
We re-type every delivery note into our system. Can you help us automate that? | 0.98 | Build one automation | 96% | 0.2 | Write a proposal for: Build one automation
Our invoices are three weeks behind and suppliers are chasing us. Can you automate our invoice processing? | 0.99 | Build one automation | 78% | 1.9 | URGENT. Write a proposal for: Build one automation
We have four automations from last year and nobody to look after them. | 0.90 | Support | 99% | 0.3 | Write a proposal for: Support
Not sure where to start with automation. What would you suggest? | 0.89 | Automation audit | 93% | 0.1 | Write a proposal for: Automation audit
Can you build us a mobile app? | 0.93 | Not something we do | 100% | 0.1 | A person replies. No model writes anything
CHEAP SEO!!! We will get you to number 1 on Google | 0.01 | Not something we do | 100% | 0.1 | Save as spam. Nobody is pinged
asdf asdf asdf | 0.03 | Not something we do | 69% | 0.1 | Save as spam. Nobody is pinged
Can you help us with our processes? | 0.82 | Automation audit | 67% | 0.1 | Write a proposal for: Automation audit
Hello, quick question about what you do. | 0.64 | Automation audit | 49% | 0.1 | Write a proposal. The writing model picks the service
We'd like to automate onboarding at some point next year, and maybe get some training too. | 0.89 | Not something we do | 75% | 0.4 | A person replies. No model writes anything

Try it: set `SPAM_BELOW = 0.9` and run the cell again. More enquiries are dropped, and the model was never
called. Then put it back. A lost lead costs more than reading one spam message.

## Where this is in the course

- `automations/toolkit/decide.py` wraps the call above in three small helpers: `yes_no`, `choice` and `score`.
- `automations/04_speed_to_lead.py` asks these three questions about every enquiry, then routes it as in step 4.
- `tools/triage.py` runs one enquiry from the command line.